# Portfolio-safe version

This notebook is a sanitized portfolio adaptation of the author's MSc Data Analytics project.
Environment-specific paths, cloud bucket names, notebook outputs, and exact patient examples
have been removed or generalized. The original methodology and core code structure are preserved.

**Data note:** the underlying TCIA imaging/clinical data are not redistributed in this repository.
Configure your own authorized/local dataset paths before running the notebook.


In [ ]:
# --- Radiomics JSON Consolidation Notebook ---

import os
import json
import pandas as pd
import gcsfs
from tqdm.notebook import tqdm  # progress bar for notebooks

# Authenticate with Google Cloud (only required if running in Colab)
print("Authenticated with Google Cloud")

# --- Configuration ---
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "your-project-id")
BUCKET_PATH = os.getenv("GCS_RADIOMICS_PATH", "your-gcs-bucket/radiomics-output")

# Initialize GCS filesystem client
fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

# --- List all JSON files in the bucket ---
all_files = fs.find(BUCKET_PATH)
json_files = [f for f in all_files if f.endswith("features.json")]
print(f"Found {len(json_files)} features.json files in the bucket")

# --- Function to read a single JSON file and extract structured metadata ---
def parse_json_from_gcs(path):
    """
    Reads a radiomics JSON file from Google Cloud Storage and extracts:
    - PatientID, StudyID, SeriesID from the folder structure
    - Only numeric radiomics features
    """
    with fs.open(path, 'r') as f:
        data = json.load(f)

    # Extract folder hierarchy identifiers
    parts = path.split("/")
    patient_id = parts[2]     # Example: STS_XXX
    study_id   = parts[3]     # Example: 09-03-2000-NA-THIGH-48623
    series_id  = parts[4]     # Example: 10.000000-AXIAL SE T2 FAT SAT - RESEARCH-62438

    # Keep only numeric radiomics features
    features = {k: v for k, v in data.items() if isinstance(v, (int, float))}

    # Create a structured record
    record = {
        "PatientID": patient_id,
        "StudyID": study_id,
        "SeriesID": series_id
    }
    record.update(features)
    return record

# --- Read all JSON files with a progress bar ---
records = []
for path in tqdm(json_files, desc="Reading radiomics features from bucket"):
    try:
        rec = parse_json_from_gcs(path)
        records.append(rec)
    except Exception as e:
        print(f"Warning: could not read {path} due to {e}")

# --- Build consolidated DataFrame ---
df_radiomics = pd.DataFrame(records)

print(f"Consolidated {len(df_radiomics)} series with {df_radiomics.shape[1] - 3} numeric radiomics features each")
print("Unique patients:", df_radiomics['PatientID'].nunique())
print("Example rows:\n", df_radiomics[['PatientID', 'StudyID', 'SeriesID']].head())

# --- Save final CSV ---
os.makedirs("data", exist_ok=True)
output_filename = "data/03_Consolidated_radiomics_all_series.csv"
df_radiomics.to_csv(output_filename, index=False)
print(f"{output_filename} generated successfully without redundant features.json references")
